# Résultats des entraînements

Courbes tirées des `log.txt` des runs (format MimicKit, aussi écrit par `rl/train.py`, une ligne toutes les
50 itérations). On peut relancer le notebook à tout moment, y compris pendant un run : il ne fait que lire
les logs. Les runs sans `log.txt` sont ignorés.

| Run | Code | Dossier |
|---|---|---|
| Phase 3 | PPO de MimicKit (`amp/run.py`) | `outputs/phase3/ppo_heading` |
| Run 1 | PPO de `rl/` | `outputs/rl/ppo_heading` |
| Runs 2-4 | AMP de `rl/` : Walk, Run, Locomotion | `outputs/rl/amp_walk`, `amp_run`, `amp_locomotion` |

À lancer depuis la racine du repo, avec le kernel du venv du projet.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

PROJECT = Path.cwd()
EP_STEPS = 600                                   # 20 s at 30 Hz

RUNS = {                                         # label -> run dir; the order fixes the colors
    "PPO, MimicKit (phase 3)": "outputs/phase3/ppo_heading",
    "PPO, rl/ (run 1)": "outputs/rl/ppo_heading",
    "AMP Walk (run 2)": "outputs/rl/amp_walk",
    "AMP Run (run 3)": "outputs/rl/amp_run",
    "AMP Locomotion (run 4)": "outputs/rl/amp_locomotion",
}
COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]   # categorical, fixed order
INK, GRID = "#3a3a38", "#e4e3df"


def read_log(path):
    """MimicKit's text log: a header line, then one row of values per output iteration."""
    tokens = Path(path).read_text().split()
    ncol = next(i for i, t in enumerate(tokens) if _is_number(t))
    header, values = tokens[:ncol], tokens[ncol:]
    nrow = len(values) // ncol
    data = np.array(values[:nrow * ncol], dtype=float).reshape(nrow, ncol)
    return {k: data[:, i] for i, k in enumerate(header)}


def _is_number(s):
    try:
        float(s)
        return True
    except ValueError:
        return False


def style(ax, xlabel, ylabel):
    ax.set_xlabel(xlabel, color=INK)
    ax.set_ylabel(ylabel, color=INK)
    ax.grid(True, color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.tick_params(colors=INK)

## Logs disponibles

In [ ]:
logs, colors = {}, {}
for (label, run_dir), color in zip(RUNS.items(), COLORS):
    path = PROJECT / run_dir / "log.txt"
    if not path.exists():
        print(f"{label:26s} pas encore de log.txt")
        continue
    log = logs[label] = read_log(path)
    colors[label] = color
    print(f"{label:26s} {len(log['Samples']):3d} lignes, {log['Samples'][-1] / 1e6:5.0f} M échantillons, "
          f"{log['Wall_Time'][-1]:.2f} h, retour normalisé {log['Test_Return'][-1] / EP_STEPS:.3f}, "
          f"épisodes {log['Test_Episode_Length'][-1] / 30:.1f} s")

## Comparaison des runs : épisodes de test pendant l'entraînement

Retour de tâche normalisé = somme de r_G sur l'épisode / 600 (entre 0 et 1 ; pour les runs AMP, r_G seul,
donc comparable au PPO seul). Durée des épisodes : 20 s au maximum.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
for label, log in logs.items():
    s = log["Samples"] / 1e6
    axes[0].plot(s, log["Test_Return"] / EP_STEPS, color=colors[label], linewidth=2, label=label)
    axes[1].plot(s, log["Test_Episode_Length"] / 30, color=colors[label], linewidth=2, label=label)
style(axes[0], "échantillons (M)", "retour de tâche normalisé")
style(axes[1], "échantillons (M)", "durée des épisodes de test (s)")
axes[0].set_ylim(0, 1)
axes[1].set_ylim(0, 20.5)
axes[0].legend(frameon=False, loc="lower right")
fig.tight_layout()
fig.savefig(PROJECT / "outputs" / "runs_comparison.png", dpi=110)
plt.show()

## PPO de MimicKit contre PPO de `rl/`

Même expérience (phase 3 et run 1) : le décollage et le plateau doivent coïncider, au bruit près
(une seule graine de chaque côté).

In [ ]:
def first_crossing(log, col, value):
    """Samples (M) at the first log row where col >= value, or None."""
    idx = np.nonzero(log[col] >= value)[0]
    return log["Samples"][idx[0]] / 1e6 if len(idx) else None

for label in ("PPO, MimicKit (phase 3)", "PPO, rl/ (run 1)"):
    if label not in logs:
        print(f"{label:26s} pas encore de log")
        continue
    log = logs[label]
    t10 = first_crossing(log, "Test_Episode_Length", 10 * 30)
    last = slice(-5, None)                     # mean of the last 5 rows (~30 M samples)
    print(f"{label:26s} épisodes > 10 s à {'%.0f M' % t10 if t10 else 'pas encore'}, "
          f"fin : retour {log['Test_Return'][last].mean() / EP_STEPS:.3f}, "
          f"épisodes {log['Test_Episode_Length'][last].mean() / 30:.1f} s, "
          f"{log['Samples_Per_Second'][-1]:.0f} échantillons/s")

## Runs AMP : discriminateur et récompenses

Repères (`rl/README.md`) : précisions de D entre ~0,55 et ~0,95 (collées à 1 : D a gagné, r_S tombe à 0 ;
collées à 0,5 : D n'apprend rien), r_S qui baisse au début puis remonte, pénalité de gradient petite et
stable, r_G qui monte comme en PPO, sans doute plus lentement.

In [ ]:
amp_runs = {label: log for label, log in logs.items() if "Disc_Acc_Mocap" in log}
if not amp_runs:
    print("pas encore de run AMP")

for label, log in amp_runs.items():
    s = log["Samples"] / 1e6
    fig, axes = plt.subplots(2, 2, figsize=(13, 8))
    panels = [
        (axes[0, 0], [("Task_Reward_Mean", "r_G (tâche)"), ("Style_Reward_Mean", "r_S (style)")],
         "récompense moyenne par pas", (0, 1)),
        (axes[0, 1], [("Disc_Acc_Mocap", "mocap jugée mocap"), ("Disc_Acc_Agent", "agent jugé agent")],
         "précision de D", (0, 1.02)),
        (axes[1, 0], [("Disc_Loss", "perte LSGAN (éq. 6)")], "perte de D", None),
        (axes[1, 1], [("Disc_Grad_Penalty", "pénalité de gradient (éq. 8)")], "pénalité de gradient", None),
    ]
    for ax, cols, ylabel, ylim in panels:
        for (col, name), color in zip(cols, COLORS):
            if col in log:
                ax.plot(s, log[col], color=color, linewidth=2, label=name)
        style(ax, "échantillons (M)", ylabel)
        if ylim:
            ax.set_ylim(*ylim)
        if len(cols) > 1:
            ax.legend(frameon=False)
        else:
            ax.set_title(cols[0][1], color=INK, fontsize=10)
    axes[0, 1].axhline(0.5, color="#8a8980", linewidth=1, linestyle="--")
    fig.suptitle(label, color=INK)
    fig.tight_layout()
    fig.savefig(PROJECT / RUNS[label] / "amp_curves.png", dpi=110)
    plt.show()